In [1]:
corpus = """
the king ruled the kingdom and the queen ruled the castle
a brave knight served the king and a wise wizard served the queen
the knight fought the dragon and the wizard cast a spell
the king gave gold to the knight and the queen gave silver to the wizard
a dark dragon burned the village and a brave knight saved the village
the wizard found a magic sword and gave the sword to the knight
the queen sent the wizard to the mountain and the king sent the knight to the forest
the dragon lived in the mountain and the knight lived in the castle
a wise queen ruled the kingdom and a brave king fought the dragon
the knight took the sword and fought the dark dragon
the wizard cast a spell and saved the king from the dragon
the queen gave gold to the wizard and the king gave silver to the knight
the village feared the dragon and the kingdom honored the knight
a magic spell saved the kingdom and the brave knight slayed the dragon
the king and the queen ruled the kingdom together
"""

In [2]:
words = corpus.lower().split()
vocab = sorted(set(words))

VOCAB_SIZE = len(vocab)

print(f"Vocab size:", VOCAB_SIZE)

Vocab size: 39


In [3]:
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}
tokens = [word_to_idx[w] for w in words]
idx_to_word

{0: 'a',
 1: 'and',
 2: 'brave',
 3: 'burned',
 4: 'cast',
 5: 'castle',
 6: 'dark',
 7: 'dragon',
 8: 'feared',
 9: 'forest',
 10: 'fought',
 11: 'found',
 12: 'from',
 13: 'gave',
 14: 'gold',
 15: 'honored',
 16: 'in',
 17: 'king',
 18: 'kingdom',
 19: 'knight',
 20: 'lived',
 21: 'magic',
 22: 'mountain',
 23: 'queen',
 24: 'ruled',
 25: 'saved',
 26: 'sent',
 27: 'served',
 28: 'silver',
 29: 'slayed',
 30: 'spell',
 31: 'sword',
 32: 'the',
 33: 'to',
 34: 'together',
 35: 'took',
 36: 'village',
 37: 'wise',
 38: 'wizard'}

In [4]:
import torch

SEQ_LEN = 8  # context window

X, Y = [], []
for i in range(len(tokens) - SEQ_LEN):
    X.append(tokens[i:i+SEQ_LEN])
    Y.append(tokens[i+1:i+1+SEQ_LEN])  # shifted by 1

X = torch.tensor(X)  # (num_samples, SEQ_LEN)
Y = torch.tensor(Y)  # (num_samples, SEQ_LEN)
print(X.shape, Y.shape)

torch.Size([181, 8]) torch.Size([181, 8])


In [9]:
from torch import nn

tok_emb = nn.Embedding(VOCAB_SIZE, 32)   # (25, 32) matrix
pos_emb = nn.Embedding(SEQ_LEN, 32)      # (8, 32) matrix

# say one training input is:
x = X[0]  # shape: (8,) — 8 token indices

tok_vectors = tok_emb(x)                          # (8, 32) — look up each word
pos_vectors = pos_emb(torch.arange(SEQ_LEN))      # (8, 32) — look up each position

embedding = tok_vectors + pos_vectors              # (8, 32) — combined
print(embedding.shape)  # torch.Size([8, 32])

torch.Size([8, 32])


In [10]:
import torch.nn.functional as F

Q_proj = nn.Linear(32, 32, bias=False)
K_proj = nn.Linear(32, 32, bias=False)
V_proj = nn.Linear(32, 32, bias=False)

# project embeddings
q = Q_proj(embedding)  # (8, 32)
k = K_proj(embedding)  # (8, 32)
v = V_proj(embedding)  # (8, 32)

# score: how relevant is each token to every other?
scores = q @ k.transpose(-2, -1) / (32 ** 0.5)  # (8, 8)

# causal mask: token i can only see tokens 0..i
mask = torch.tril(torch.ones(8, 8))  # lower triangular
scores = scores.masked_fill(mask == 0, float('-inf'))

# normalize to weights
weights = F.softmax(scores, dim=-1)  # (8, 8) — each row sums to 1

# weighted sum of values
output = weights @ v  # (8, 32)

In [12]:
ff = nn.Sequential(
    nn.Linear(32, 64),   # expand
    nn.ReLU(),
    nn.Linear(64, 32),   # compress back
)

ln1 = nn.LayerNorm(32)
ln2 = nn.LayerNorm(32)

# residual connections: add input back to output
x = embedding + output           # skip connection around attention
x = ln1(x)                       # normalize
x = x + ff(x)                    # skip connection around feed-forward
x = ln2(x)                       # normalize again
print(x.shape)  # (8, 32)

torch.Size([8, 32])


In [13]:
head = nn.Linear(32, VOCAB_SIZE)

logits = head(x)  # (8, VOCAB_SIZE) — raw scores for each word at each position
print(logits.shape)

# grab the target for this training pair
y = Y[0]  # (8,) — the actual next tokens

# cross-entropy loss: how wrong were the predictions?
loss = F.cross_entropy(logits, y)
print(loss)

torch.Size([8, 39])
tensor(3.7048, grad_fn=<NllLossBackward0>)


In [14]:
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, 32)
        self.pos_emb = nn.Embedding(SEQ_LEN, 32)
        self.Q = nn.Linear(32, 32, bias=False)
        self.K = nn.Linear(32, 32, bias=False)
        self.V = nn.Linear(32, 32, bias=False)
        self.ff = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 32))
        self.ln1 = nn.LayerNorm(32)
        self.ln2 = nn.LayerNorm(32)
        self.head = nn.Linear(32, VOCAB_SIZE)

    def forward(self, idx):
        B, T = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok + pos

        q, k, v = self.Q(x), self.K(x), self.V(x)
        scores = q @ k.transpose(-2, -1) / (32 ** 0.5)
        mask = torch.tril(torch.ones(T, T, device=idx.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        x = x + attn @ v
        x = self.ln1(x)
        x = x + self.ff(x)
        x = self.ln2(x)
        return self.head(x)

In [15]:
model = MiniGPT()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(500):
    logits = model(X)                    # (num_samples, 8, VOCAB_SIZE)
    loss = F.cross_entropy(
        logits.view(-1, VOCAB_SIZE),     # flatten to (num_samples*8, VOCAB_SIZE)
        Y.view(-1)                       # flatten to (num_samples*8,)
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"epoch {epoch}, loss: {loss.item():.4f}")

epoch 0, loss: 3.8068
epoch 50, loss: 2.5243
epoch 100, loss: 1.7509
epoch 150, loss: 1.2990
epoch 200, loss: 1.0125
epoch 250, loss: 0.7590
epoch 300, loss: 0.5833
epoch 350, loss: 0.4854
epoch 400, loss: 0.4219
epoch 450, loss: 0.3782


In [17]:
print(sum(p.numel() for p in model.parameters()))

def generate(model, prompt, max_new_tokens=10):
    model.eval()
    tokens = [word_to_idx[w] for w in prompt.lower().split()]
    
    for _ in range(max_new_tokens):
        x = torch.tensor([tokens[-SEQ_LEN:]])       # take last SEQ_LEN tokens
        logits = model(x)
        next_logit = logits[0, -1, :]                # last position's prediction
        next_token = torch.argmax(next_logit).item()  # greedy pick
        tokens.append(next_token)
    
    return ' '.join(idx_to_word[t] for t in tokens)

print(generate(model, "the king"))
print(generate(model, "the wizard cast"))
print(generate(model, "a brave knight"))

10183
the king gave silver to the knight the village feared the dragon
the wizard cast a spell the king gave gold to the knight and
a brave knight served the king and a wise wizard served the queen
